# Qchem 扩展代码功能验证演示 Notebook

这个 notebook 不是简单 import 检查，而是按“**后端要看采样/状态向量、拟设要看实例化线路图、算法要看实际数值结果并与理论/精确值对比**”来写。

使用方式：

1. 先把 `qchem_extension` 里的 `.py` 文件复制到原 Qchem 源库的对应目录；
2. 不要覆盖旧的 `__init__.py`，只合并新增导入；
3. 把本 notebook 放在 Qchem 仓库根目录运行；
4. 推荐执行 `Kernel -> Restart Kernel and Run All Cells`。

每一节都对应一个新增 `.py` 文件或一个新增功能入口。可选后端 `Cirq / Qulacs / QuTiP` 如果没有安装，会显示 `SKIPPED`，核心 NumPy 模拟、拟设和算法演示仍会继续。

## 0. 环境准备与公共绘图/打印工具

这一节做三件事：

- 确认当前路径是 Qchem 仓库根目录；
- 提供 `draw_with_compiler()`：优先调用项目里的 `backend.compiler.draw(circuit)` 或 `backends.compiler.draw(circuit)`，如果当前项目没有这个接口，则退化为 ASCII 线路图；
- 提供统一的状态向量、采样和数值比较打印函数。

In [1]:

import os
import sys
import math
import importlib
import traceback
from pathlib import Path

import numpy as np

try:
    import torch
except Exception:
    torch = None

ROOT = Path.cwd()
print("当前工作目录:", ROOT)
print("建议目录中应存在 backends/ ansatz/ algorithms/ 三个文件夹。")
print("存在性检查:", {p: (ROOT / p).exists() for p in ["backends", "ansatz", "algorithms"]})

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DEMO_RESULTS = []

def mark(name, status="PASS", detail=""):
    DEMO_RESULTS.append({"name": name, "status": status, "detail": detail})
    icon = "✅" if status == "PASS" else ("⚠️" if status == "SKIPPED" else "❌")
    print(f"{icon} {name}: {status} {detail}")


def as_numpy(x):
    if torch is not None and hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def nonzero_amplitudes(state, n_qubits, atol=1e-9):
    state = as_numpy(state).reshape(-1)
    rows = []
    for idx, amp in enumerate(state):
        if abs(amp) > atol:
            rows.append((format(idx, f"0{n_qubits}b"), amp))
    return rows


def print_statevector(state, n_qubits, title="Statevector", atol=1e-9):
    print(f"\n{title}")
    print("-" * len(title))
    for bit, amp in nonzero_amplitudes(state, n_qubits, atol=atol):
        print(f"|{bit}> : {amp.real:+.8f}{amp.imag:+.8f}j")
    norm = np.linalg.norm(as_numpy(state).reshape(-1))
    print(f"norm = {norm:.12f}")


def print_probabilities(state, n_qubits, title="Probabilities"):
    state = as_numpy(state).reshape(-1)
    probs = np.abs(state) ** 2
    print(f"\n{title}")
    print("-" * len(title))
    for idx, p in enumerate(probs):
        if p > 1e-9:
            print(f"P({format(idx, f'0{n_qubits}b')}) = {p:.8f}")
    print(f"sum(P) = {probs.sum():.12f}")


def circuit_to_ascii(circuit):
    """Fallback ASCII drawer when project compiler.draw is unavailable."""
    n = getattr(circuit, "n_qubits", None)
    instructions = list(getattr(circuit, "instructions", []))
    if n is None:
        return str(circuit)
    lines = [[f"q{i}: "] for i in range(n)]
    for k, inst in enumerate(instructions):
        name = str(getattr(inst, "name", type(inst).__name__))
        qubits = list(getattr(inst, "qubits", []) or [])
        params = list(getattr(inst, "params", []) or [])
        label = name
        if params:
            short_params = []
            for p in params[:2]:
                try:
                    val = float(p.detach().cpu().numpy()) if hasattr(p, "detach") else float(p)
                    short_params.append(f"{val:.3f}")
                except Exception:
                    short_params.append(str(p))
            label += "(" + ",".join(short_params) + ")"
        width = max(len(label) + 2, 8)
        for q in range(n):
            if q in qubits:
                lines[q].append(label.center(width, "-"))
            else:
                lines[q].append("-" * width)
    return "\n".join("".join(x) for x in lines)


def draw_with_compiler(circuit, backend=None, title="Circuit"):
    """优先调用项目里的 compiler.draw；没有则退化为 ASCII 图。"""
    print(f"\n=== {title} ===")
    # 1) backend.compiler.draw(circuit)
    try:
        compiler = getattr(backend, "compiler", None) if backend is not None else None
        if compiler is not None and hasattr(compiler, "draw"):
            print("使用 backend.compiler.draw(circuit) 输出线路图：")
            out = compiler.draw(circuit)
            try:
                from IPython.display import display
                display(out)
            except Exception:
                print(out)
            return out
    except Exception as exc:
        print("backend.compiler.draw 调用失败，转用 fallback。原因:", repr(exc))
    # 2) backends.compiler.draw(circuit)
    try:
        compiler_module = importlib.import_module("backends.compiler")
        if hasattr(compiler_module, "draw"):
            print("使用 backends.compiler.draw(circuit) 输出线路图：")
            out = compiler_module.draw(circuit)
            try:
                from IPython.display import display
                display(out)
            except Exception:
                print(out)
            return out
    except Exception as exc:
        print("未检测到 backends.compiler.draw，使用 ASCII fallback。")
    # 3) fallback
    art = circuit_to_ascii(circuit)
    print(art)
    return art


def make_bell_circuit():
    from backends.core import QuantumCircuit
    qc = QuantumCircuit(2, name="BellStateDemo")
    qc.h(0)
    qc.cx(0, 1)
    return qc


def assert_close(name, value, expected, tol=1e-8):
    value = np.asarray(value)
    expected = np.asarray(expected)
    err = float(np.max(np.abs(value - expected)))
    print(f"{name}: max|diff| = {err:.3e}")
    assert err <= tol, f"{name} not close: {err} > {tol}"

当前工作目录: e:\xiaoming\HYQ-ALG-LIB-new
建议目录中应存在 backends/ ansatz/ algorithms/ 三个文件夹。
存在性检查: {'backends': True, 'ansatz': True, 'algorithms': True}


## 1. `backends/_matrix_simulator.py`：状态向量、概率、采样、Pauli 期望值

特征验证：用 Bell 线路 `H(0) -> CX(0,1)` 制备

\[
|\Phi^+
angle = (|00
angle + |11
angle)/\sqrt{2}
\]

因此理论状态向量只有 `|00>` 和 `|11>` 两项，采样结果也应该只出现 `00` 和 `11`，而且比例约为 1:1。

In [2]:

from backends._matrix_simulator import (
    simulate_statevector,
    probability_distribution,
    sample_counts,
    expectation_from_pauli_terms,
)

qc = make_bell_circuit()
draw_with_compiler(qc, title="Bell 线路：H(0) + CX(0,1)")

state = simulate_statevector(qc)
expected = np.zeros(4, dtype=np.complex128)
expected[0] = 1 / np.sqrt(2)
expected[3] = 1 / np.sqrt(2)

print_statevector(state, 2, "_matrix_simulator 输出状态向量")
print_probabilities(state, 2, "_matrix_simulator 输出概率分布")
print("采样结果 shots=2000:", sample_counts(state, shots=2000, measure_qubits=[0, 1], seed=123))

# Bell 态满足 <ZZ> = 1, <XX> = 1
zz = {((0, "Z"), (1, "Z")): 1.0}
xx = {((0, "X"), (1, "X")): 1.0}
print("<ZZ> =", expectation_from_pauli_terms(state, zz, 2))
print("<XX> =", expectation_from_pauli_terms(state, xx, 2))

assert_close("Bell statevector", state, expected, tol=1e-8)
assert abs(expectation_from_pauli_terms(state, zz, 2).real - 1.0) < 1e-8
assert abs(expectation_from_pauli_terms(state, xx, 2).real - 1.0) < 1e-8
mark("backends/_matrix_simulator.py", "PASS", "Bell 态状态向量、采样、<ZZ>/<XX> 均符合理论。")

Please first ``pip install -U qiskit`` to enable related functionality in translation module



=== Bell 线路：H(0) + CX(0,1) ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---H------CNOT--
q1: ----------CNOT--

_matrix_simulator 输出状态向量
------------------------
|00> : +0.70710678+0.00000000j
|11> : +0.70710678+0.00000000j
norm = 1.000000000000

_matrix_simulator 输出概率分布
------------------------
P(00) = 0.50000000
P(11) = 0.50000000
sum(P) = 1.000000000000
采样结果 shots=2000: {'11': 978, '00': 1022}
<ZZ> = (1.0000000000000002+0j)
<XX> = (1.0000000000000002+0j)
Bell statevector: max|diff| = 1.110e-16
✅ backends/_matrix_simulator.py: PASS Bell 态状态向量、采样、<ZZ>/<XX> 均符合理论。


## 2. `backends/Cirq.py`：Cirq 后端状态向量与采样模式

特征验证：同样运行 Bell 线路，打印 Cirq 后端返回的状态向量和采样结果。Cirq 官方模拟文档说明其内置模拟器支持 pure state 和 mixed state 两类模拟，本测试使用 pure-state `cirq.Simulator`。

In [3]:

try:
    from backends.Cirq import CirqBackend
    backend = CirqBackend(seed=2026)
    qc = make_bell_circuit()
    draw_with_compiler(qc, backend=backend, title="CirqBackend 上的 Bell 线路")
    sv = backend.get_statevector(qc)
    counts = backend.run_sampling(qc, shots=2000, measure_qubits=[0, 1])
    print_statevector(sv, 2, "CirqBackend.get_statevector 输出")
    print("CirqBackend.run_sampling 输出 counts:", counts)
    assert_close("CirqBackend vs reference", as_numpy(sv), state, tol=1e-8)
    assert set(counts).issubset({"00", "11"})
    mark("backends/Cirq.py", "PASS", "状态向量与 NumPy 参考一致，采样只出现 00/11。")
except ImportError as exc:
    mark("backends/Cirq.py", "SKIPPED", f"未安装 cirq：{exc}")
except Exception as exc:
    mark("backends/Cirq.py", "FAIL", repr(exc))
    traceback.print_exc()


=== CirqBackend 上的 Bell 线路 ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---H------CNOT--
q1: ----------CNOT--

CirqBackend.get_statevector 输出
------------------------------
|00> : +0.70710678+0.00000000j
|11> : +0.70710678+0.00000000j
norm = 1.000000000000
CirqBackend.run_sampling 输出 counts: {'00': 969, '11': 1031}
CirqBackend vs reference: max|diff| = 0.000e+00
✅ backends/Cirq.py: PASS 状态向量与 NumPy 参考一致，采样只出现 00/11。


## 3. `backends/Qulacs.py`：Qulacs 后端状态向量与采样模式

特征验证：用 Qulacs 后端执行 Bell 线路。Qulacs 本身定位为高性能 C++/Python 量子线路模拟库；这里同时展示其 native statevector 路径和统一采样输出。

In [4]:

try:
    from backends.Qulacs import QulacsBackend
    backend = QulacsBackend(seed=2026, use_native=True)
    qc = make_bell_circuit()
    draw_with_compiler(qc, backend=backend, title="QulacsBackend 上的 Bell 线路")
    sv = backend.get_statevector(qc)
    counts = backend.run_sampling(qc, shots=2000, measure_qubits=[0, 1])
    print_statevector(sv, 2, "QulacsBackend.get_statevector 输出")
    print("QulacsBackend.run_sampling 输出 counts:", counts)
    assert_close("QulacsBackend vs reference", as_numpy(sv), state, tol=1e-8)
    assert set(counts).issubset({"00", "11"})
    mark("backends/Qulacs.py", "PASS", "状态向量与 NumPy 参考一致，采样只出现 00/11。")
except ImportError as exc:
    mark("backends/Qulacs.py", "SKIPPED", f"未安装 qulacs：{exc}")
except Exception as exc:
    mark("backends/Qulacs.py", "FAIL", repr(exc))
    traceback.print_exc()


=== QulacsBackend 上的 Bell 线路 ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---H------CNOT--
q1: ----------CNOT--

QulacsBackend.get_statevector 输出
--------------------------------
|00> : +0.70710678+0.00000000j
|11> : +0.70710678+0.00000000j
norm = 1.000000000000
QulacsBackend.run_sampling 输出 counts: {'00': 983, '11': 1017}
QulacsBackend vs reference: max|diff| = 1.110e-16
✅ backends/Qulacs.py: PASS 状态向量与 NumPy 参考一致，采样只出现 00/11。


## 4. `backends/Qutip.py`：QuTiP 后端状态向量、采样与 Qobj 转换

特征验证：QuTiP 适合物理层/open-system 分析。此处先用统一 dense simulator 跑出状态向量和采样；若本地安装了 `qutip`，额外展示 `to_qobj()` 返回的 QuTiP ket 对象。

In [5]:

try:
    from backends.Qutip import QutipBackend
    # require_qutip=False 允许在没有 qutip 的环境中仍验证统一状态向量/采样接口。
    backend = QutipBackend(seed=2026, require_qutip=False)
    qc = make_bell_circuit()
    draw_with_compiler(qc, backend=backend, title="QutipBackend 上的 Bell 线路")
    sv = backend.get_statevector(qc)
    counts = backend.run_sampling(qc, shots=2000, measure_qubits=[0, 1])
    print_statevector(sv, 2, "QutipBackend.get_statevector 输出")
    print("QutipBackend.run_sampling 输出 counts:", counts)
    assert_close("QutipBackend vs reference", as_numpy(sv), state, tol=1e-8)
    assert set(counts).issubset({"00", "11"})
    try:
        qobj = backend.to_qobj(qc)
        print("QuTiP Qobj dims:", qobj.dims)
        print("QuTiP Qobj shape:", qobj.shape)
    except ImportError:
        print("未安装 qutip，因此跳过 to_qobj() 展示。安装 qutip 后会显示 Qobj dims/shape。")
    mark("backends/Qutip.py", "PASS", "状态向量与 NumPy 参考一致，采样只出现 00/11。")
except Exception as exc:
    mark("backends/Qutip.py", "FAIL", repr(exc))
    traceback.print_exc()


=== QutipBackend 上的 Bell 线路 ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---H------CNOT--
q1: ----------CNOT--

QutipBackend.get_statevector 输出
-------------------------------
|00> : +0.70710678+0.00000000j
|11> : +0.70710678+0.00000000j
norm = 1.000000000000
QutipBackend.run_sampling 输出 counts: {'00': 983, '11': 1017}
QutipBackend vs reference: max|diff| = 0.000e+00
QuTiP Qobj dims: [[2, 2], [1]]
QuTiP Qobj shape: (4, 1)
✅ backends/Qutip.py: PASS 状态向量与 NumPy 参考一致，采样只出现 00/11。


## 5. `backends/extended_registry.py`：后端注册表

特征验证：检查新增后端是否出现在注册表，并尝试通过 `get_backend()` 创建后端对象。可选依赖未安装时，创建对象本身通常可以成功，真正执行时才会提示安装。

In [6]:

try:
    from backends.extended_registry import available_backends, get_backend, register_backend
    names = available_backends()
    print("available_backends() =", names)
    for name in ["cirq", "qulacs", "qutip"]:
        print(f"{name} in registry?", name in names)
    b = get_backend("qutip", require_qutip=False)
    print("get_backend('qutip') ->", type(b).__name__)
    assert "cirq" in names and "qulacs" in names and "qutip" in names
    mark("backends/extended_registry.py", "PASS", "新增后端已进入注册表，并可创建 backend 实例。")
except Exception as exc:
    mark("backends/extended_registry.py", "FAIL", repr(exc))
    traceback.print_exc()

available_backends() = ['cirq', 'qiskit', 'qulacs', 'qutip', 'tensorcircuit']
cirq in registry? True
qulacs in registry? True
qutip in registry? True
get_backend('qutip') -> QutipBackend
✅ backends/extended_registry.py: PASS 新增后端已进入注册表，并可创建 backend 实例。


## 6. `ansatz/ryrz.py`：RyRz 通用拟设

特征验证：

- 实例化 3 比特、2 层、`alternating` 纠缠、指定初态 `bitstring:100`；
- 打印参数总数和每层纠缠边；
- 用 `draw_with_compiler()` 画线路；
- 用状态向量模拟验证线路可执行，输出最终波函数和概率分布。

In [7]:

try:
    from ansatz.ryrz import RyRzAnsatz, entangler_pairs
    from backends._matrix_simulator import simulate_statevector

    ans = RyRzAnsatz(
        n_qubits=3,
        n_layers=2,
        entanglement="alternating",
        initial_state="bitstring:100",
        final_layer=True,
    )
    params = np.linspace(0.05, 0.05 * ans.n_params, ans.n_params)
    circuit = ans.forward(params)

    print("RyRzAnsatz 参数数量 n_params =", ans.n_params)
    print("第0层 alternating 纠缠边:", entangler_pairs(3, "alternating", 0))
    print("第1层 alternating 纠缠边:", entangler_pairs(3, "alternating", 1))
    print("线路指令数 =", len(circuit.instructions))
    draw_with_compiler(circuit, title="RyRzAnsatz(n_qubits=3, n_layers=2, entanglement='alternating')")

    sv = simulate_statevector(circuit)
    print_statevector(sv, 3, "RyRzAnsatz 输出状态向量（非零振幅）", atol=1e-5)
    print_probabilities(sv, 3, "RyRzAnsatz 输出概率分布")
    assert abs(np.linalg.norm(sv) - 1.0) < 1e-8
    mark("ansatz/ryrz.py", "PASS", "完成实例化、画图和状态向量模拟。")
except Exception as exc:
    mark("ansatz/ryrz.py", "FAIL", repr(exc))
    traceback.print_exc()

RyRzAnsatz 参数数量 n_params = 27
第0层 alternating 纠缠边: [(0, 1)]
第1层 alternating 纠缠边: [(1, 2)]
线路指令数 = 30

=== RyRzAnsatz(n_qubits=3, n_layers=2, entanglement='alternating') ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---X-----RZ(0.050)--RY(0.100)--RZ(0.150)---------------------------------------------------------------------CNOT---RZ(0.500)--RY(0.550)--RZ(0.600)----------------------------------------------------------------------------RZ(0.950)--RY(1.000)--RZ(1.050)-------------------------------------------------------------------
q1: ------------------------------------------RZ(0.200)--RY(0.250)--RZ(0.300)------------------------------------CNOT------------------------------------RZ(0.650)--RY(0.700)--RZ(0.750)------------------------------------CNOT------------------------------------RZ(1.100)--RY(1.150)--RZ(1.200)----------------------------------
q2: ---------------------------------------------------------------------------RZ(0.350)--RY(0.400)--RZ(0.450)------------------

## 7. `ansatz/compact.py`：紧致 PairExcitation 与 k-UpCCG 拟设

特征验证：

- `PairExcitationAnsatz`：4 比特 2 电子，只有少量 pair-transfer 参数，体现“紧致”；
- `KUpCCGAnsatz`：展示 k 层 generalized pair coupled-cluster 风格线路；
- 两个线路都画图并运行状态向量模拟；
- 检查粒子数期望接近 2，体现闭壳层/电子对初态特点。

In [8]:

try:
    from ansatz.compact import PairExcitationAnsatz, KUpCCGAnsatz
    from algorithms.quantum_chemistry import particle_number_expectation
    from backends._matrix_simulator import simulate_statevector

    pair_ans = PairExcitationAnsatz(n_qubits=4, n_electrons=2, repetitions=1, include_pair_phase=True)
    pair_params = np.array([0.25, -0.10], dtype=float)
    pair_circuit = pair_ans.forward(pair_params)
    print("PairExcitationAnsatz n_params =", pair_ans.n_params)
    print("Pair excitations =", pair_ans.excitations)
    print("线路指令数 =", len(pair_circuit.instructions))
    draw_with_compiler(pair_circuit, title="PairExcitationAnsatz(n_qubits=4, n_electrons=2)")
    pair_sv = simulate_statevector(pair_circuit)
    print_statevector(pair_sv, 4, "PairExcitationAnsatz 输出状态向量", atol=1e-5)
    print("粒子数期望 <N> =", particle_number_expectation(pair_sv, 4))

    kupccg = KUpCCGAnsatz(n_qubits=4, n_electrons=2, k=2, include_orbital_rotations=True)
    kupccg_params = np.linspace(0.03, 0.03 * kupccg.n_params, kupccg.n_params)
    kupccg_circuit = kupccg.forward(kupccg_params)
    print("\nKUpCCGAnsatz n_params =", kupccg.n_params)
    print("pair_moves =", kupccg.pair_moves)
    print("线路指令数 =", len(kupccg_circuit.instructions))
    draw_with_compiler(kupccg_circuit, title="KUpCCGAnsatz(n_qubits=4, n_electrons=2, k=2)")
    kupccg_sv = simulate_statevector(kupccg_circuit)
    print_statevector(kupccg_sv, 4, "KUpCCGAnsatz 输出状态向量", atol=1e-5)
    print("粒子数期望 <N> =", particle_number_expectation(kupccg_sv, 4))

    assert pair_ans.n_params < 10
    assert kupccg.n_params < 20
    mark("ansatz/compact.py", "PASS", "PairExcitation/k-UpCCG 均完成实例化、画图和状态向量模拟。")
except Exception as exc:
    mark("ansatz/compact.py", "FAIL", repr(exc))
    traceback.print_exc()

PairExcitationAnsatz n_params = 2
Pair excitations = [PairExcitation(source_orbital=0, target_orbital=1, alpha_source=0, beta_source=1, alpha_target=2, beta_target=3)]
线路指令数 = 14

=== PairExcitationAnsatz(n_qubits=4, n_electrons=2) ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---X--------------CNOT---------------CNOT------------------------------------------RZZ(-0.250)--------------------------RZ(0.100)------------
q1: -----------X---------------------------------CNOT---------------CNOT---------------RZZ(-0.250)-------------------------------------RZ(0.100)-
q2: ------------------CNOT---RY(0.250)---CNOT------------------------------RZZ(0.250)---------------RZ(-0.100)-----------------------------------
q3: ---------------------------------------------CNOT---RY(0.250)---CNOT---RZZ(0.250)---------------------------RZ(-0.100)-----------------------

PairExcitationAnsatz 输出状态向量
---------------------------
|1100> : +0.96483263+0.19558126j
|1101> : -0.11620226-0.04241714j
|1110> : -

## 8. `ansatz/adapt.py`：ADAPT-VQE 风格自适应拟设

特征验证：

- 生成 UCCSD-like operator pool 和 pair-only pool；
- 用一组模拟梯度选择最大梯度算符，展示 ADAPT “选择最大梯度算符加入线路”的特点；
- 实例化 `ADAPTAnsatz`，画出新增算符产生的线路；
- 用有限差分函数验证梯度工具是否工作。

In [9]:

try:
    from ansatz.adapt import (
        ADAPTAnsatz,
        build_uccsd_operator_pool,
        build_pair_operator_pool,
        finite_difference_gradient,
        select_largest_gradient,
    )
    from backends._matrix_simulator import simulate_statevector

    pool = build_uccsd_operator_pool(n_qubits=4, n_electrons=2, include_singles=True, include_doubles=False)
    pair_pool = build_pair_operator_pool(n_qubits=4, n_electrons=2)
    print("UCCSD-like singles pool size =", len(pool))
    print("Pair-only pool size =", len(pair_pool))
    print("前3个 pool 算符标签:", [op.label for op in pool[:3]])

    fake_gradients = np.linspace(0.01, 0.20, len(pool))
    selected, max_grad, converged = select_largest_gradient(fake_gradients, pool, threshold=1e-4)
    print("选中的最大梯度算符:", selected.label)
    print("max_grad =", max_grad, "converged?", converged)

    adapt_ans = ADAPTAnsatz(n_qubits=4, n_electrons=2, selected_operators=[selected], trotter_steps=1)
    params = np.array([0.18])
    adapt_circuit = adapt_ans.forward(params)
    print("ADAPTAnsatz n_params =", adapt_ans.n_params)
    print("线路指令数 =", len(adapt_circuit.instructions))
    draw_with_compiler(adapt_circuit, title=f"ADAPTAnsatz selected operator = {selected.label}")

    sv = simulate_statevector(adapt_circuit)
    print_statevector(sv, 4, "ADAPTAnsatz 输出状态向量", atol=1e-5)

    # finite_difference_gradient 的独立数值验证：f(x)=(x-0.3)^2 在 x=0.1 处导数应为 -0.4
    params_t = torch.tensor([0.1], dtype=torch.float64) if torch is not None else np.array([0.1])
    def toy_energy(x):
        val = x[0]
        if hasattr(val, "item"):
            val = val.item()
        return (float(val) - 0.3) ** 2
    grad = finite_difference_gradient(toy_energy, params_t, operator_index=0, epsilon=1e-5)
    print("有限差分梯度 demo: f(x)=(x-0.3)^2 at x=0.1")
    print("计算梯度 =", grad, "理论梯度 = -0.4")
    assert abs(grad + 0.4) < 1e-4
    mark("ansatz/adapt.py", "PASS", "完成 pool 构造、最大梯度选择、线路生成和有限差分梯度验证。")
except Exception as exc:
    mark("ansatz/adapt.py", "FAIL", repr(exc))
    traceback.print_exc()

UCCSD-like singles pool size = 4
Pair-only pool size = 1
前3个 pool 算符标签: ['S(0->2)', 'S(0->3)', 'S(1->2)']
选中的最大梯度算符: S(1->3)
max_grad = 0.2 converged? False
ADAPTAnsatz n_params = 1
线路指令数 = 20

=== ADAPTAnsatz selected operator = S(1->3) ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---X---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
q1: -----------X-------H-----------------CNOT-------------------------------CNOT-----------------H-----RX(1.571)-----------CNOT--------------------------------CNOT-----------RX(-1.571)-
q2: -------------------------------------CNOT----CNOT---------------CNOT----CNOT-------------------------------------------CNOT----CNOT----------------CNOT----CNOT----------------------
q3: -------------------------RX(1.571)-----------CNOT---RZ(0.180)---CNOT-----------RX(-1.571)-----------------------H--------------CNOT---RZ(-0.180)---CN

## 9. `algorithms/trotter.py`：Trotter 时间演化

特征验证：

对单比特哈密顿量 \(H=X\) 做时间演化

\[
U(t)=e^{-iXt}, \quad t=\pi/2
\]

理论上 \(U|0
angle=-i|1
angle\)，所以测量 `|1>` 的概率应为 1。这里构造 Trotter 线路、画图、运行状态向量，并与理论概率对比。

In [10]:

try:
    from algorithms.trotter import first_order_trotter_circuit, second_order_trotter_circuit, trotter_info
    from backends._matrix_simulator import simulate_statevector

    Hx = [(((0, "X"),), 1.0)]
    t = np.pi / 2
    circ1 = first_order_trotter_circuit(Hx, time=t, n_steps=1, n_qubits=1)
    info1 = trotter_info(Hx, time=t, n_steps=1, order=1)
    print("First-order Trotter info:", info1)
    draw_with_compiler(circ1, title="First-order Trotter circuit for exp(-i X pi/2)")
    sv1 = simulate_statevector(circ1)
    print_statevector(sv1, 1, "Trotter 输出状态向量")
    print_probabilities(sv1, 1, "Trotter 输出概率")
    print("理论: |1> 概率 = 1.0；实际:", abs(sv1[1]) ** 2)
    assert abs(abs(sv1[1]) ** 2 - 1.0) < 1e-8

    # 二阶 Trotter 对单项哈密顿量应与一阶等价
    circ2 = second_order_trotter_circuit(Hx, time=t, n_steps=1, n_qubits=1)
    sv2 = simulate_statevector(circ2)
    assert abs(abs(sv2[1]) ** 2 - 1.0) < 1e-8
    mark("algorithms/trotter.py", "PASS", "exp(-i X pi/2)|0> 得到 |1>，与理论一致。")
except Exception as exc:
    mark("algorithms/trotter.py", "FAIL", repr(exc))
    traceback.print_exc()

First-order Trotter info: TrotterStepInfo(order=1, time=1.5707963267948966, n_steps=1, n_terms=1, gate_count_estimate=3)

=== First-order Trotter circuit for exp(-i X pi/2) ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ---H-----RZ(3.142)----H----

Trotter 输出状态向量
--------------
|1> : +0.00000000-1.00000000j
norm = 1.000000000000

Trotter 输出概率
------------
P(1) = 1.00000000
sum(P) = 1.000000000000
理论: |1> 概率 = 1.0；实际: 1.0
✅ algorithms/trotter.py: PASS exp(-i X pi/2)|0> 得到 |1>，与理论一致。


## 10. `algorithms/classical_eigensolvers.py`：精确对角化、Lanczos、Rayleigh 商、HF 初态

特征验证：

对单比特哈密顿量

\[
H = Z + 0.5X
\]

理论本征值为 \(\pm\sqrt{1+0.5^2}=\pm\sqrt{1.25}\)。这里用 dense exact diagonalization 和 Lanczos 分别计算最低能量，并与理论值对比。

In [11]:

try:
    from algorithms.classical_eigensolvers import (
        exact_diagonalization,
        lanczos_lowest_eigenvalue,
        hamiltonian_matrix,
        rayleigh_quotient,
        reduced_density_matrix,
        hartree_fock_bitstring,
        hartree_fock_state,
    )

    H = [(((0, "Z"),), 1.0), (((0, "X"),), 0.5)]
    mat = hamiltonian_matrix(H, n_qubits=1)
    exact = exact_diagonalization(H, n_qubits=1)
    lanczos = lanczos_lowest_eigenvalue(H, n_qubits=1, seed=7)
    theory_low = -np.sqrt(1.25)
    theory_high = np.sqrt(1.25)
    print("Hamiltonian matrix =\n", mat)
    print("Exact eigenvalues =", exact.eigenvalues)
    print("Lanczos lowest eigenvalue =", lanczos.eigenvalues[0])
    print("理论本征值 =", [theory_low, theory_high])
    assert abs(exact.eigenvalues[0] - theory_low) < 1e-8
    assert abs(lanczos.eigenvalues[0] - theory_low) < 1e-8

    gs = exact.eigenvectors[:, 0]
    print("Rayleigh quotient of exact ground vector =", rayleigh_quotient(mat, gs))
    print("HF bitstring n_qubits=4, n_electrons=2 ->", hartree_fock_bitstring(4, 2))
    hf = hartree_fock_state(4, 2)
    print_statevector(hf, 4, "Hartree-Fock computational basis state")
    bell = np.zeros(4, dtype=np.complex128); bell[0] = bell[3] = 1 / np.sqrt(2)
    print("Bell 态单比特约化密度矩阵 =\n", reduced_density_matrix(bell, keep=[0], n_qubits=2))
    mark("algorithms/classical_eigensolvers.py", "PASS", "exact/Lanczos 均复现实验哈密顿量理论本征值。")
except Exception as exc:
    mark("algorithms/classical_eigensolvers.py", "FAIL", repr(exc))
    traceback.print_exc()

❌ algorithms/classical_eigensolvers.py: FAIL TypeError('Cannot use scipy.linalg.eig for sparse A with k >= N - 1. Use scipy.linalg.eig(A.toarray()) or reduce k.')


c:\Users\xiaoming\miniconda3\Lib\site-packages\scipy\sparse\linalg\_eigen\arpack\arpack.py:1581: RuntimeWarning: k >= N - 1 for N * N square matrix. Attempting to use scipy.linalg.eig instead.
  ret = eigs(A, k, M=M, sigma=sigma, which=which, v0=v0,
Traceback (most recent call last):
  File "C:\Users\xiaoming\AppData\Local\Temp\ipykernel_54456\3035525932.py", line 15, in <module>
    lanczos = lanczos_lowest_eigenvalue(H, n_qubits=1, seed=7)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\xiaoming\HYQ-ALG-LIB-new\algorithms\classical_eigensolvers.py", line 120, in lanczos_lowest_eigenvalue
    vals, vecs = spla.eigsh(mat, k=1, which="SA", maxiter=maxiter, tol=tol)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\xiaoming\miniconda3\Lib\site-packages\scipy\sparse\linalg\_eigen\arpack\arpack.py", line 1581, in eigsh
    ret = eigs(A, k, M=M, sigma=sigma, which=which, v0=v0,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

## 11. `algorithms/phase_estimation.py`：QPE 相位估计工具

特征验证：

构造一个 1 比特对角酉矩阵

\[
U = \mathrm{diag}(1, e^{2\pi i\phi}), \quad \phi=5/8=0.625
\]

输入本征态 `|1>`，使用 3 个辅助比特时理论上应该集中到二进制 `101`，也就是相位 0.625。

In [ ]:

try:
    from algorithms.phase_estimation import (
        exact_phase_estimation_from_unitary,
        phases_from_counts,
        iterative_phase_estimation_step,
        qft,
        inverse_qft,
    )
    from backends.core import QuantumCircuit

    phi = 5 / 8
    U = np.diag([1.0, np.exp(2j * np.pi * phi)])
    input_state = np.array([0.0, 1.0], dtype=np.complex128)
    pe = exact_phase_estimation_from_unitary(U, input_state, n_ancilla=3)
    best = int(np.argmax(pe.probabilities))
    print("QPE grid bitstrings:", pe.bitstrings)
    print("QPE probabilities:", pe.probabilities)
    print("最高概率 bitstring =", pe.bitstrings[best], "phase =", pe.phases[best])
    print("理论 bitstring = 101, 理论 phase = 0.625")
    assert pe.bitstrings[best] == "101"
    assert abs(pe.phases[best] - 0.625) < 1e-12

    counts = {"101": 990, "100": 10}
    parsed = phases_from_counts(counts, n_ancilla=3)
    print("phases_from_counts demo:", list(zip(parsed.bitstrings, parsed.phases, parsed.probabilities)))

    phase_from_hadamard = iterative_phase_estimation_step(
        expectation_cos=np.cos(2 * np.pi * phi),
        expectation_sin=np.sin(2 * np.pi * phi),
    )
    print("iterative_phase_estimation_step 输出 =", phase_from_hadamard)
    assert abs(phase_from_hadamard - phi) < 1e-12

    qft_circ = QuantumCircuit(3, name="QFT-then-IQFT-demo")
    qft(qft_circ, [0, 1, 2])
    inverse_qft(qft_circ, [0, 1, 2])
    draw_with_compiler(qft_circ, title="QFT + inverse QFT 线路结构展示")
    mark("algorithms/phase_estimation.py", "PASS", "QPE 对 phi=5/8 输出最高概率 bitstring=101，与理论一致。")
except Exception as exc:
    mark("algorithms/phase_estimation.py", "FAIL", repr(exc))
    traceback.print_exc()

QPE grid bitstrings: ['000', '001', '010', '011', '100', '101', '110', '111']
QPE probabilities: [0. 0. 0. 0. 0. 1. 0. 0.]
最高概率 bitstring = 101 phase = 0.625
理论 bitstring = 101, 理论 phase = 0.625
phases_from_counts demo: [('101', 0.625, 0.99), ('100', 0.5, 0.01)]
iterative_phase_estimation_step 输出 = 0.625

=== QFT + inverse QFT 线路结构展示 ===
未检测到 backends.compiler.draw，使用 ASCII fallback。
q0: ------------------------CPhase(0.785)----------CPhase(1.571)----H------SWAP----SWAP-----H-----CPhase(-1.571)----------CPhase(-0.785)-------------------------
q1: ---------CPhase(1.571)-------------------H-----CPhase(1.571)----------------------------------CPhase(-1.571)----H---------------------CPhase(-1.571)---------
q2: ---H-----CPhase(1.571)--CPhase(0.785)----------------------------------SWAP----SWAP-----------------------------------CPhase(-0.785)--CPhase(-1.571)----H----
✅ algorithms/phase_estimation.py: PASS QPE 对 phi=5/8 输出最高概率 bitstring=101，与理论一致。


: 

## 12. `algorithms/qse.py`：Quantum Subspace Expansion

特征验证：

以 \(H=Z\) 为例，参考态取 `|0>`，激发算符取 `X`，则子空间包含 `|0>` 和 `|1>`，QSE 应该恢复 `Z` 的两个本征值 `[-1, +1]`（顺序从低到高）。

In [ ]:

try:
    from algorithms.qse import quantum_subspace_expansion, singles_doubles_qse_pool

    ref = np.array([1.0, 0.0], dtype=np.complex128)  # |0>
    H = [(((0, "Z"),), 1.0)]
    excitations = [[(((0, "X"),), 1.0)]]
    qse_res = quantum_subspace_expansion(ref, H, excitations, n_qubits=1, regularization=1e-10)
    print("QSE overlap matrix S =\n", qse_res.overlap_matrix)
    print("QSE effective Hamiltonian H_eff =\n", qse_res.hamiltonian_matrix)
    print("QSE energies =", qse_res.energies)
    print("理论 energies = [-1, +1]")
    assert np.allclose(np.sort(qse_res.energies), np.array([-1.0, 1.0]), atol=1e-6)

    pool = singles_doubles_qse_pool(n_qubits=4, n_electrons=2)
    print("singles_doubles_qse_pool(4,2) size =", len(pool))
    print("第一个 QSE pool 元素 =", pool[0] if pool else None)
    mark("algorithms/qse.py", "PASS", "QSE 子空间恢复 H=Z 的精确本征值。")
except Exception as exc:
    mark("algorithms/qse.py", "FAIL", repr(exc))
    traceback.print_exc()

## 13. `algorithms/vqd.py`：VQD 激发态 deflation 工具

特征验证：

VQD 的核心是给已找到的低能态加罚项。令

\[
H=\mathrm{diag}(-1, +1)
\]

基态为 `|0>`，第一激发态为 `|1>`。如果加入较大的 deflation 罚项 \(eta |0
angle\langle 0|\)，新的最低本征态应转移到原来的激发态，最低能量应变为 `+1`。

In [ ]:

try:
    from algorithms.vqd import state_overlap, vqd_objective, gram_schmidt_states, deflated_matrix

    H = np.diag([-1.0, 1.0]).astype(np.complex128)
    ground = np.array([1.0, 0.0], dtype=np.complex128)
    excited = np.array([0.0, 1.0], dtype=np.complex128)
    beta = 3.0
    H_def = deflated_matrix(H, previous_states=[ground], betas=[beta])
    evals, evecs = np.linalg.eigh(H_def)
    print("原 H eigenvalues =", np.linalg.eigvalsh(H))
    print("Deflated H =\n", H_def)
    print("Deflated eigenvalues =", evals)
    print("理论：加罚后最低能量应为 +1，对应原第一激发态 |1>")
    assert abs(evals[0] - 1.0) < 1e-12

    print("overlap(excited, ground) =", state_overlap(excited, ground))
    print("overlap(ground, ground) =", state_overlap(ground, ground))
    obj_ground = vqd_objective(-1.0, ground, [ground], betas=[beta])
    obj_excited = vqd_objective(1.0, excited, [ground], betas=[beta])
    print("VQD objective for old ground:", obj_ground)
    print("VQD objective for excited state:", obj_excited)
    basis = gram_schmidt_states([ground, ground + excited, excited])
    print("Gram-Schmidt basis size =", len(basis))
    mark("algorithms/vqd.py", "PASS", "Deflation 将最低态从旧基态转移到原激发态，符合 VQD 理论。")
except Exception as exc:
    mark("algorithms/vqd.py", "FAIL", repr(exc))
    traceback.print_exc()

## 14. `algorithms/quantum_chemistry.py`：量子化学参考与分析工具

特征验证：

- `fci_reference_energy`：对小哈密顿量做精确最低能量参考；
- `chemical_accuracy_error`：输出 Hartree 与 kcal/mol 误差，并判断是否在化学精度内；
- `state_fidelity`：比较同态和正交态；
- `mp2_energy_from_integrals`：用 toy MO 能级和积分计算 MP2 correlation energy；
- `particle_number_expectation / spin_z_expectation / symmetry_project_state`：检查粒子数和自旋投影相关工具。

In [ ]:

try:
    from algorithms.quantum_chemistry import (
        mp2_energy_from_integrals,
        fci_reference_energy,
        chemical_accuracy_error,
        state_fidelity,
        natural_orbital_occupations,
        particle_number_expectation,
        spin_z_expectation,
        active_space_indices,
        freeze_core_energy_shift,
        symmetry_project_state,
        commutator_norm,
    )
    from algorithms.classical_eigensolvers import hamiltonian_matrix

    # FCI reference on a toy qubit Hamiltonian H = -Z，理论最低能量 -1
    H = [(((0, "Z"),), -1.0)]
    fci = fci_reference_energy(H, n_qubits=1)
    print("fci_reference_energy for H=-Z =", fci, "理论 = -1")
    assert abs(fci + 1.0) < 1e-12

    err = chemical_accuracy_error(estimated_energy=-0.999, reference_energy=-1.0)
    print("chemical_accuracy_error(-0.999, -1.0) =", err)

    s0 = np.array([1.0, 0.0], dtype=np.complex128)
    s1 = np.array([0.0, 1.0], dtype=np.complex128)
    print("fidelity(|0>, |0>) =", state_fidelity(s0, s0))
    print("fidelity(|0>, |1>) =", state_fidelity(s0, s1))
    assert abs(state_fidelity(s0, s0).fidelity - 1.0) < 1e-12
    assert abs(state_fidelity(s0, s1).fidelity - 0.0) < 1e-12

    # toy MP2: 两占据两虚轨道，设置一个非零双电子积分，输出 correlation energy
    eps = np.array([-1.0, -0.8, 0.3, 0.5])
    eri = np.zeros((4, 4, 4, 4))
    eri[0, 1, 2, 3] = 0.05
    eri[0, 1, 3, 2] = 0.02
    eri[1, 0, 2, 3] = 0.05
    eri[1, 0, 3, 2] = 0.02
    mp2 = mp2_energy_from_integrals(eps, eri, n_electrons=2, hf_energy=-1.10)
    print("MP2 toy result =", mp2)
    print("说明：toy 积分只用于验证公式路径，真实分子需由 Psi4/PySCF/OpenFermion 提供积分。")

    hf_like = np.zeros(16, dtype=np.complex128)
    hf_like[int("1100", 2)] = 1.0
    print("particle_number_expectation(|1100>) =", particle_number_expectation(hf_like, 4))
    print("spin_z_expectation(|1100>) =", spin_z_expectation(hf_like, 4))
    projected = symmetry_project_state(hf_like, n_qubits=4, n_particles=2, sz=0.0)
    print_statevector(projected, 4, "按 n_particles=2, Sz=0 投影后的态")
    print("active_space_indices(n_orbitals=8, n_core=1, n_active=4) =", active_space_indices(8, n_core=1, n_active=4))
    print("freeze_core_energy_shift([-20.0]) =", freeze_core_energy_shift([-20.0]))
    one_rdm = np.diag([1.9, 1.8, 0.2, 0.1])
    print("natural_orbital_occupations =", natural_orbital_occupations(one_rdm))

    X = np.array([[0,1],[1,0]], dtype=complex)
    Z = np.array([[1,0],[0,-1]], dtype=complex)
    print("commutator_norm(X, Z) =", commutator_norm(X, Z), "理论 = ||XZ-ZX|| = sqrt(8)")
    assert abs(commutator_norm(X, Z) - np.sqrt(8)) < 1e-12
    mark("algorithms/quantum_chemistry.py", "PASS", "FCI、化学精度、保真度、MP2、粒子数/自旋/投影工具均完成演示。")
except Exception as exc:
    mark("algorithms/quantum_chemistry.py", "FAIL", repr(exc))
    traceback.print_exc()

## 15. `__init__.py` 合并后的包级导入检查

这一节用于验证你是否正确“合并”了 `ansatz/__init__.py`、`backends/__init__.py`、`algorithms/__init__.py` 的新增导入。

如果这里失败，不代表功能文件本身坏了，通常是 `__init__.py` 没有把新增类/函数导出来。

In [ ]:

try:
    import ansatz
    import backends
    import algorithms

    expected_ansatz_names = ["RyRzAnsatz", "PairExcitationAnsatz", "KUpCCGAnsatz", "ADAPTAnsatz"]
    expected_backend_names = ["CirqBackend", "QulacsBackend", "QutipBackend"]
    expected_algorithm_names = [
        "exact_diagonalization", "first_order_trotter_circuit", "exact_phase_estimation_from_unitary",
        "quantum_subspace_expansion", "vqd_objective", "fci_reference_energy"
    ]

    print("ansatz package exports:")
    for name in expected_ansatz_names:
        print(f"  {name}:", hasattr(ansatz, name))

    print("\nbackends package exports:")
    for name in expected_backend_names:
        print(f"  {name}:", hasattr(backends, name))

    print("\nalgorithms package exports:")
    for name in expected_algorithm_names:
        print(f"  {name}:", hasattr(algorithms, name))

    missing = []
    missing += ["ansatz." + n for n in expected_ansatz_names if not hasattr(ansatz, n)]
    missing += ["backends." + n for n in expected_backend_names if not hasattr(backends, n)]
    missing += ["algorithms." + n for n in expected_algorithm_names if not hasattr(algorithms, n)]
    if missing:
        print("\n缺少这些包级导出，请检查对应 __init__.py 是否合并了新增导入：")
        for x in missing:
            print(" -", x)
        mark("__init__.py integration", "FAIL", "缺少包级导出：" + ", ".join(missing))
    else:
        mark("__init__.py integration", "PASS", "新增类/函数均可从包级入口导入。")
except Exception as exc:
    mark("__init__.py integration", "FAIL", repr(exc))
    traceback.print_exc()

## 16. 总结表

运行到这里后，检查 `status`：

- `PASS`：该文件的核心功能已完成演示；
- `SKIPPED`：通常是可选依赖未安装，例如 `cirq/qulacs/qutip`；
- `FAIL`：需要根据报错回到对应文件或 `__init__.py` 修复。

In [ ]:

try:
    import pandas as pd
    df = pd.DataFrame(DEMO_RESULTS)
    display(df)
    print("\n统计:")
    print(df["status"].value_counts())
except Exception:
    print(DEMO_RESULTS)